# Extended news regressions — zero-fill specification

This notebook is a robustness/production version after `01_market_sector_news_patch.ipynb`.

Key difference from the common-sample version:
- missing news indices are interpreted as **no observed news signal** and filled with `0`;
- if count columns exist (`n_firm`, `n_market`, `n_sector`), they are filled with `0` and may be used as intensity controls;
- models are compared on the same full ticker-level sample after market variables are available.

This is useful because dropping all days without firm/market/sector news can reduce the sample too aggressively.


In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

DATA_PATH = "data/final_features_daily_extended_contexts.parquet"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

print(df.shape)
print(df.columns.tolist())
df.head()


(2512, 32)
['ticker', 'date', 'returns', 'RSI', 'MACD', 'I_t', 'I_t_max', 'N_t_strong', 'n_news', 'I_t_pos', 'I_t_neg', 'I_firm', 'I_firm_max_abs', 'n_firm', 'N_firm_strong', 'p_pos_firm', 'p_neg_firm', 'confidence_firm', 'I_market', 'I_market_max_abs', 'n_market', 'N_market_strong', 'p_pos_market', 'p_neg_market', 'confidence_market', 'I_sector', 'I_sector_max_abs', 'n_sector', 'N_sector_strong', 'p_pos_sector', 'p_neg_sector', 'confidence_sector']


,ticker,date,returns,RSI,MACD,I_t,I_t_max,N_t_strong,n_news,I_t_pos,...,p_pos_market,p_neg_market,confidence_market,I_sector,I_sector_max_abs,n_sector,N_sector_strong,p_pos_sector,p_neg_sector,confidence_sector
0,AAPL,2020-12-29,NaN,NaN,0.000000,-0.527073,0.527073,1.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
1,AAPL,2020-12-30,-0.008527,0.000000,-0.089302,-0.122154,0.878942,2.0,2.0,0.634633,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
2,AAPL,2020-12-31,-0.007703,0.000000,-0.238232,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
3,AAPL,2021-01-04,-0.024719,0.000000,-0.606907,0.659435,0.892991,1.0,2.0,0.659435,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
4,AAPL,2021-01-05,0.012364,8.684289,-0.764590,-0.162034,0.510277,1.0,3.0,0.060650,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN


## Prepare zero-filled news indices

In [3]:
NEWS_COLS = ["I_firm", "I_market", "I_sector"]
COUNT_COLS_CANDIDATES = [
    "n_firm", "n_market", "n_sector",
    "n_news_firm", "n_news_market", "n_news_sector",
]

# Backward compatibility: if old firm index is still named I_t, copy it to I_firm.
if "I_firm" not in df.columns and "I_t" in df.columns:
    df["I_firm"] = df["I_t"]

for col in NEWS_COLS:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")
    df[col] = df[col].fillna(0.0)

count_cols = [c for c in COUNT_COLS_CANDIDATES if c in df.columns]
for col in count_cols:
    df[col] = df[col].fillna(0.0)

# Construct dependent and control variables if needed.

# Ensure r_log exists. In the current pipeline the input parquet may contain
# returns but not r_log/r_log_fwd1.
if "r_log" not in df.columns:
    if "returns" in df.columns:
        df["returns"] = pd.to_numeric(df["returns"], errors="coerce")
        df["r_log"] = np.log1p(df["returns"])
    else:
        price_candidates = ["adj_close", "Adj Close", "adjclose", "close", "Close"]
        price_col = next((c for c in price_candidates if c in df.columns), None)
        if price_col is None:
            raise ValueError(
                "Expected r_log, returns, or price column. Existing columns: "
                f"{df.columns.tolist()[:40]}"
            )
        df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
        df["r_log"] = df.groupby("ticker")[price_col].transform(lambda s: np.log(s).diff())

# Target: next-day return.
if "r_log_fwd1" not in df.columns:
    df["r_log_fwd1"] = df.groupby("ticker")["r_log"].shift(-1)

# Lagged return control.
if "r_log_lag1" not in df.columns:
    df["r_log_lag1"] = df.groupby("ticker")["r_log"].shift(1)

# Optional changes in technical indicators.
if "d_RSI" not in df.columns and "RSI" in df.columns:
    df["d_RSI"] = df.groupby("ticker")["RSI"].diff()

if "d_macd" not in df.columns and "MACD" in df.columns:
    df["d_macd"] = df.groupby("ticker")["MACD"].diff()

base_controls = [c for c in ["r_log_lag1", "RSI", "MACD"] if c in df.columns]
print("Base controls:", base_controls)
print("Count cols:", count_cols)
df[["ticker", "date", "r_log_fwd1"] + NEWS_COLS + base_controls].head()


Base controls: ['r_log_lag1', 'RSI', 'MACD']
Count cols: ['n_firm', 'n_market', 'n_sector']


,ticker,date,r_log_fwd1,I_firm,I_market,I_sector,r_log_lag1,RSI,MACD
0,AAPL,2020-12-29,-0.008563,-0.527073,0.0,0.0,NaN,NaN,0.000000
1,AAPL,2020-12-30,-0.007732,-0.122154,0.0,0.0,NaN,0.000000,-0.089302
2,AAPL,2020-12-31,-0.025030,0.000000,0.0,0.0,-0.008563,0.000000,-0.238232
3,AAPL,2021-01-04,0.012288,0.659435,0.0,0.0,-0.007732,0.000000,-0.606907
4,AAPL,2021-01-05,-0.034241,-0.162034,0.0,0.0,-0.025030,8.684289,-0.764590


## Coverage after zero-fill

In [4]:
coverage_rows = []
for ticker, g in df.groupby("ticker"):
    row = {"ticker": ticker, "n_total": len(g)}
    for col in NEWS_COLS:
        row[f"n_{col}_original_nonmissing"] = int(g[col].notna().sum())
        row[f"share_{col}_available_after_zerofill"] = float(g[col].notna().mean())
        row[f"share_{col}_nonzero"] = float((g[col].abs() > 1e-12).mean())
    coverage_rows.append(row)

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(os.path.join(OUT_DIR, "coverage_extended_news_indices_zerofill.csv"), index=False)
coverage


,ticker,n_total,n_I_firm_original_nonmissing,share_I_firm_available_after_zerofill,share_I_firm_nonzero,n_I_market_original_nonmissing,share_I_market_available_after_zerofill,share_I_market_nonzero,n_I_sector_original_nonmissing,share_I_sector_available_after_zerofill,share_I_sector_nonzero
0,AAPL,1256,1256,1.0,0.579618,1256,1.0,0.664013,1256,1.0,0.725318
1,XOM,1256,1256,1.0,0.496019,1256,1.0,0.661624,1256,1.0,0.761943


## Correlations

In [5]:
corr_rows = []
targets = [c for c in ["r_log_fwd1", "d_RSI", "d_macd"] if c in df.columns]
for ticker, g in df.groupby("ticker"):
    for y in targets:
        for x in NEWS_COLS:
            sub = g[[y, x]].replace([np.inf, -np.inf], np.nan).dropna()
            corr = sub[y].corr(sub[x]) if len(sub) > 2 else np.nan
            corr_rows.append({"ticker": ticker, "y": y, "x": x, "n": len(sub), "corr": corr})

corr = pd.DataFrame(corr_rows)
corr.to_csv(os.path.join(OUT_DIR, "corr_extended_news_indices_zerofill.csv"), index=False)
corr


,ticker,y,x,n,corr
0,AAPL,r_log_fwd1,I_firm,1255,-0.000012
1,AAPL,r_log_fwd1,I_market,1255,0.008411
2,AAPL,r_log_fwd1,I_sector,1255,-0.046390
3,AAPL,d_RSI,I_firm,1254,0.056367
4,AAPL,d_RSI,I_market,1254,0.125867
5,AAPL,d_RSI,I_sector,1254,-0.007255
6,AAPL,d_macd,I_firm,1255,0.059547
7,AAPL,d_macd,I_market,1255,0.133446
8,AAPL,d_macd,I_sector,1255,0.030755
9,XOM,r_log_fwd1,I_firm,1255,0.027183


## OLS with HAC standard errors

In [6]:
def fit_ols_hac(g, y, xcols, hac_lags=5):
    sub = g[[y] + xcols].replace([np.inf, -np.inf], np.nan).dropna()
    if len(sub) < max(30, len(xcols) + 5):
        return None
    X = sm.add_constant(sub[xcols], has_constant="add")
    model = sm.OLS(sub[y], X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    rows = []
    for term in model.params.index:
        rows.append({
            "model": None,
            "y": y,
            "term": term,
            "coef": model.params[term],
            "std_err_HAC": model.bse[term],
            "t": model.tvalues[term],
            "p_value": model.pvalues[term],
            "nobs": int(model.nobs),
            "r2": model.rsquared,
            "aic": model.aic,
            "bic": model.bic,
        })
    return rows

specs = {
    "M0_controls_only": base_controls,
    "M1_firm": base_controls + ["I_firm"],
    "M2_firm_market": base_controls + ["I_firm", "I_market"],
    "M3_firm_market_sector": base_controls + ["I_firm", "I_market", "I_sector"],
}

# Optional: add news intensity controls as an additional robustness spec.
if count_cols:
    specs["M4_news_plus_counts"] = base_controls + NEWS_COLS + count_cols

ols_rows = []
for ticker, g in df.groupby("ticker"):
    for name, xcols in specs.items():
        rows = fit_ols_hac(g, "r_log_fwd1", xcols)
        if rows is None:
            continue
        for r in rows:
            r["model"] = name
            r["ticker"] = ticker
        ols_rows.extend(rows)

ols = pd.DataFrame(ols_rows)
ols.to_csv(os.path.join(OUT_DIR, "ols_extended_news_indices_zerofill.csv"), index=False)
ols


,model,y,term,coef,std_err_HAC,t,p_value,nobs,r2,aic,bic,ticker
0,M0_controls_only,r_log_fwd1,const,-0.002408,0.002963,-0.812550,0.416476,1253,0.001894,-6576.073173,-6555.539989,AAPL
1,M0_controls_only,r_log_fwd1,r_log_lag1,-0.007850,0.035512,-0.221060,0.825046,1253,0.001894,-6576.073173,-6555.539989,AAPL
2,M0_controls_only,r_log_fwd1,RSI,0.000062,0.000058,1.084776,0.278021,1253,0.001894,-6576.073173,-6555.539989,AAPL
3,M0_controls_only,r_log_fwd1,MACD,-0.000391,0.000252,-1.550069,0.121125,1253,0.001894,-6576.073173,-6555.539989,AAPL
4,M1_firm,r_log_fwd1,const,-0.002407,0.002965,-0.811853,0.416876,1253,0.001895,-6574.073347,-6548.406867,AAPL
...,...,...,...,...,...,...,...,...,...,...,...,...
59,M4_news_plus_counts,r_log_fwd1,I_market,-0.000649,0.001143,-0.567472,0.570394,1253,0.014071,-6647.722292,-6596.389332,XOM
60,M4_news_plus_counts,r_log_fwd1,I_sector,0.000537,0.001250,0.429652,0.667449,1253,0.014071,-6647.722292,-6596.389332,XOM
61,M4_news_plus_counts,r_log_fwd1,n_firm,-0.000009,0.000009,-0.945072,0.344622,1253,0.014071,-6647.722292,-6596.389332,XOM
62,M4_news_plus_counts,r_log_fwd1,n_market,0.000080,0.000063,1.260217,0.207591,1253,0.014071,-6647.722292,-6596.389332,XOM


## Out-of-sample comparison

All models are evaluated on the same ticker-level rows. The baseline is the training-sample mean return.


In [7]:
def oos_eval_ticker(g, y, specs, test_frac=0.30):
    # Common sample across all specs to make MSE comparable.
    all_cols = sorted(set([y] + [c for cols in specs.values() for c in cols]))
    sub = g[all_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(sub) < 80:
        return []
    split = int(len(sub) * (1 - test_frac))
    train = sub.iloc[:split]
    test = sub.iloc[split:]
    y_train = train[y]
    y_test = test[y]
    baseline_pred = np.repeat(y_train.mean(), len(test))
    baseline_mse = mean_squared_error(y_test, baseline_pred)
    rows = [{
        "ticker": g["ticker"].iloc[0],
        "model": "baseline_mean",
        "n_train": len(train),
        "n_test": len(test),
        "mse": baseline_mse,
        "mse_rel_to_baseline_pct": 0.0,
    }]
    for name, xcols in specs.items():
        X_train = sm.add_constant(train[xcols], has_constant="add")
        X_test = sm.add_constant(test[xcols], has_constant="add")
        model = sm.OLS(y_train, X_train).fit()
        pred = model.predict(X_test)
        mse = mean_squared_error(y_test, pred)
        rows.append({
            "ticker": g["ticker"].iloc[0],
            "model": name,
            "n_train": len(train),
            "n_test": len(test),
            "mse": mse,
            "mse_rel_to_baseline_pct": 100 * (mse / baseline_mse - 1),
        })
    return rows

oos_rows = []
for ticker, g in df.groupby("ticker"):
    oos_rows.extend(oos_eval_ticker(g, "r_log_fwd1", specs))

oos = pd.DataFrame(oos_rows)
oos.to_csv(os.path.join(OUT_DIR, "oos_extended_news_indices_zerofill.csv"), index=False)
oos


,ticker,model,n_train,n_test,mse,mse_rel_to_baseline_pct
0,AAPL,baseline_mean,877,376,0.000330,0.000000
1,AAPL,M0_controls_only,877,376,0.000334,1.122335
2,AAPL,M1_firm,877,376,0.000334,1.132372
3,AAPL,M2_firm_market,877,376,0.000337,2.000862
4,AAPL,M3_firm_market_sector,877,376,0.000337,2.127847
5,AAPL,M4_news_plus_counts,877,376,0.000493,49.134727
6,XOM,baseline_mean,877,376,0.000201,0.000000
7,XOM,M0_controls_only,877,376,0.000203,0.750617
8,XOM,M1_firm,877,376,0.000203,0.692254
9,XOM,M2_firm_market,877,376,0.000204,1.195285


## Distributed lag robustness

In [8]:
K = 3
dl_rows = []
for ticker, g0 in df.groupby("ticker"):
    g = g0.sort_values("date").copy()
    lag_cols = []
    for col in NEWS_COLS:
        for k in range(K + 1):
            new_col = col if k == 0 else f"{col}_lag{k}"
            if k > 0:
                g[new_col] = g[col].shift(k)
            lag_cols.append(new_col)
    xcols = lag_cols + base_controls
    rows = fit_ols_hac(g, "r_log_fwd1", xcols, hac_lags=5)
    if rows is None:
        continue
    for r in rows:
        r["model"] = f"DL_K{K}_extended_zerofill"
        r["ticker"] = ticker
    dl_rows.extend(rows)

dl = pd.DataFrame(dl_rows)
dl.to_csv(os.path.join(OUT_DIR, "distributed_lag_extended_news_indices_zerofill.csv"), index=False)
dl


,model,y,term,coef,std_err_HAC,t,p_value,nobs,r2,aic,bic,ticker
0,DL_K3_extended_zerofill,r_log_fwd1,const,-0.001403,0.003033,-0.462735,0.643555,1252,0.012789,-6561.705568,-6479.585607,AAPL
1,DL_K3_extended_zerofill,r_log_fwd1,I_firm,0.000107,0.001518,0.070664,0.943665,1252,0.012789,-6561.705568,-6479.585607,AAPL
2,DL_K3_extended_zerofill,r_log_fwd1,I_firm_lag1,0.000300,0.001666,0.179913,0.857221,1252,0.012789,-6561.705568,-6479.585607,AAPL
3,DL_K3_extended_zerofill,r_log_fwd1,I_firm_lag2,0.000351,0.001529,0.229204,0.818710,1252,0.012789,-6561.705568,-6479.585607,AAPL
4,DL_K3_extended_zerofill,r_log_fwd1,I_firm_lag3,0.001807,0.001522,1.186937,0.235253,1252,0.012789,-6561.705568,-6479.585607,AAPL
5,DL_K3_extended_zerofill,r_log_fwd1,I_market,0.000381,0.001185,0.321144,0.748102,1252,0.012789,-6561.705568,-6479.585607,AAPL
6,DL_K3_extended_zerofill,r_log_fwd1,I_market_lag1,0.000230,0.001088,0.211143,0.832776,1252,0.012789,-6561.705568,-6479.585607,AAPL
7,DL_K3_extended_zerofill,r_log_fwd1,I_market_lag2,-0.001949,0.001256,-1.551645,0.120747,1252,0.012789,-6561.705568,-6479.585607,AAPL
8,DL_K3_extended_zerofill,r_log_fwd1,I_market_lag3,0.002507,0.001383,1.812994,0.069833,1252,0.012789,-6561.705568,-6479.585607,AAPL
9,DL_K3_extended_zerofill,r_log_fwd1,I_sector,-0.002049,0.001449,-1.414854,0.157111,1252,0.012789,-6561.705568,-6479.585607,AAPL
